# Simplest CRNN for Polish phoneme windows

- Mel spectrogram as input
- Classification window: **50 ms**, shift: **20 ms**
- Label for each window = **vowel with the largest time overlap** in that window; `non_vowel` if no vowel is present.
- Spectrogram hop = **10 ms** (so 1 window = 5 spectrogram frames, shift = 2 frames).

In [1]:
import os, glob
import numpy as np
import torch
import torch.nn as nn
import torchaudio.transforms as T
from torch.utils.data import Dataset, DataLoader, random_split
from scipy.io import wavfile

# ---- spectrogram config ----
SAMPLE_RATE = 16000
N_FFT       = 400            # 25 ms
HOP_LENGTH  = 160            # 10 ms   -> 1 spec frame = 10 ms
N_MELS      = 64

# ---- classification window ----
WIN_MS       = 50
SHIFT_MS     = 20
FRAME_MS     = HOP_LENGTH * 1000 // SAMPLE_RATE        # 10
WIN_FRAMES   = WIN_MS   // FRAME_MS                    # 5
SHIFT_FRAMES = SHIFT_MS // FRAME_MS                    # 2
print(f'window = {WIN_FRAMES} frames, shift = {SHIFT_FRAMES} frames')

# ---- labels: Polish vowels + non_vowel ----
VOWELS    = ['a', 'e', 'i', 'i2', 'o', 'u', 'y', 'eo5', 'oc5']
NON_VOWEL = 'non_vowel'
LABELS    = VOWELS + [NON_VOWEL]
LABEL2IDX = {l: i for i, l in enumerate(LABELS)}
IDX2LABEL = {i: l for l, i in LABEL2IDX.items()}
N_CLASSES = len(LABELS)
print('classes:', LABELS)

window = 5 frames, shift = 2 frames
classes: ['a', 'e', 'i', 'i2', 'o', 'u', 'y', 'eo5', 'oc5', 'non_vowel']


## TextGrid parser (same as in the exploration notebook)

In [2]:
def parse_phonemes(text_grid):
    """Return list of (xmin, xmax, text) from the 'phones' tier."""
    phonemes = []
    in_phones = False
    xmin = xmax = None
    for line in text_grid.split('\n'):
        line = line.strip()
        if 'name = "phones"' in line:
            in_phones = True; continue
        if in_phones and line.startswith('name =') and 'phones' not in line:
            break
        if not in_phones:
            continue
        if line.startswith('xmin =') and 'intervals' not in line:
            xmin = float(line.split('=')[1].strip())
        elif line.startswith('xmax =') and 'intervals' not in line:
            xmax = float(line.split('=')[1].strip())
        elif line.startswith('text ='):
            text = line.split('=', 1)[1].strip().strip('"')
            if xmin is not None and xmax is not None:
                phonemes.append((xmin, xmax, text))
            xmin = xmax = None
    return phonemes

## Mel spectrogram + windowing with dominant-vowel labels

In [3]:
mel_tf = T.MelSpectrogram(sample_rate=SAMPLE_RATE, n_fft=N_FFT,
                          hop_length=HOP_LENGTH, n_mels=N_MELS)
db_tf  = T.AmplitudeToDB()

def wav_to_logmel(wav_path):
    sr, audio = wavfile.read(wav_path)
    assert sr == SAMPLE_RATE, f'expected {SAMPLE_RATE} Hz, got {sr}'
    if np.issubdtype(audio.dtype, np.integer):
        audio = audio.astype(np.float32) / np.iinfo(audio.dtype).max
    else:
        audio = audio.astype(np.float32)
    wav = torch.from_numpy(audio).unsqueeze(0)  # (1, n_samples)
    mel = db_tf(mel_tf(wav)).squeeze(0)         # (n_mels, T)
    return mel


def windows_and_labels(mel, phonemes):
    """Slide a 50 ms window with 20 ms shift over the mel spectrogram.
    For each window pick the vowel with the largest overlap; else non_vowel.
    Returns a list of (window_tensor, label_idx)."""
    n_mels, T = mel.shape
    frame_dur = HOP_LENGTH / SAMPLE_RATE  # seconds per frame
    out = []
    for start in range(0, T - WIN_FRAMES + 1, SHIFT_FRAMES):
        end = start + WIN_FRAMES
        t_start = start * frame_dur
        t_end   = end   * frame_dur
        best_vowel, best_overlap = None, 0.0
        for (pmin, pmax, ptext) in phonemes:
            if ptext not in VOWELS:
                continue
            overlap = min(pmax, t_end) - max(pmin, t_start)
            if overlap > best_overlap:
                best_overlap, best_vowel = overlap, ptext
        label = best_vowel if best_vowel is not None else NON_VOWEL
        out.append((mel[:, start:end].clone(), LABEL2IDX[label]))
    return out

## Dataset — scans a directory for paired `.wav` + `.TextGrid` files

In [4]:
class PhonemeWindowDataset(Dataset):
    def __init__(self, data_dir, max_files=None, verbose=True):
        tg_paths = sorted(glob.glob(os.path.join(data_dir, '**', '*.TextGrid'),
                                    recursive=True))
        if max_files is not None:
            tg_paths = tg_paths[:max_files]
        xs, ys = [], []
        for i, tg in enumerate(tg_paths):
            wav = tg[:-len('.TextGrid')] + '.wav'
            if not os.path.exists(wav):
                continue
            try:
                mel = wav_to_logmel(wav)
                with open(tg, 'r', encoding='utf-8') as f:
                    phonemes = parse_phonemes(f.read())
                for w, lbl in windows_and_labels(mel, phonemes):
                    xs.append(w); ys.append(lbl)
            except Exception as e:
                if verbose: print(f'skip {tg}: {e}')
            if verbose and (i + 1) % 100 == 0:
                print(f'  processed {i+1}/{len(tg_paths)} files, {len(xs)} windows')
        self.X = torch.stack(xs) if xs else torch.empty(0, N_MELS, WIN_FRAMES)
        self.y = torch.tensor(ys, dtype=torch.long)
        if verbose:
            print(f'total windows: {len(self.y)}')
            counts = torch.bincount(self.y, minlength=N_CLASSES).tolist()
            for l, c in zip(LABELS, counts):
                print(f'  {l:>9}: {c}')

    def __len__(self):            return len(self.y)
    def __getitem__(self, idx):   return self.X[idx], self.y[idx]

In [5]:
DATA_DIR = '../data/501-1000'   # adjust to your dataset location

# start with max_files=200 to sanity-check; remove the limit for full training
dataset = PhonemeWindowDataset(DATA_DIR, max_files=200)
print('X shape:', dataset.X.shape, 'y shape:', dataset.y.shape)

  processed 100/200 files, 30733 windows
  processed 200/200 files, 67644 windows
total windows: 67644
          a: 7368
          e: 6187
          i: 4763
         i2: 2855
          o: 6112
          u: 2665
          y: 0
        eo5: 715
        oc5: 1411
  non_vowel: 35568
X shape: torch.Size([67644, 64, 5]) y shape: torch.Size([67644])


## CRNN model

2 conv blocks (pooling only across frequency so we keep all 5 time frames) → bi-GRU → linear.

In [6]:
class CRNN(nn.Module):
    def __init__(self, n_mels=N_MELS, n_classes=N_CLASSES, hidden=64):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16), nn.ReLU(),
            nn.MaxPool2d((2, 1)),                     # halve freq only
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d((2, 1)),                     # halve freq again
        )
        self.freq_out = n_mels // 4                    # 16
        self.rnn = nn.GRU(input_size=32 * self.freq_out,
                          hidden_size=hidden,
                          batch_first=True,
                          bidirectional=True)
        self.fc  = nn.Linear(hidden * 2, n_classes)

    def forward(self, x):
        # x: (B, n_mels, T)
        x = x.unsqueeze(1)                             # (B, 1, n_mels, T)
        x = self.conv(x)                               # (B, 32, n_mels/4, T)
        B, C, F, Tt = x.shape
        x = x.permute(0, 3, 1, 2).reshape(B, Tt, C * F)  # (B, T, C*F)
        _, h = self.rnn(x)                             # h: (2, B, hidden)
        h = torch.cat([h[0], h[1]], dim=1)             # (B, 2*hidden)
        return self.fc(h)                              # (B, n_classes)


device = 'cuda' if torch.cuda.is_available() else 'cpu'
model  = CRNN().to(device)
print(model)
print('params:', sum(p.numel() for p in model.parameters()))

CRNN(
  (conv): Sequential(
    (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=(2, 1), stride=(2, 1), padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=(2, 1), stride=(2, 1), padding=0, dilation=1, ceil_mode=False)
  )
  (rnn): GRU(512, 64, batch_first=True, bidirectional=True)
  (fc): Linear(in_features=128, out_features=10, bias=True)
)
params: 228138


## Training

In [8]:
# 80 / 20 split
n_val = max(1, int(0.2 * len(dataset)))
n_tr  = len(dataset) - n_val
train_set, val_set = random_split(dataset, [n_tr, n_val],
                                  generator=torch.Generator().manual_seed(0))

train_loader = DataLoader(train_set, batch_size=256, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_set,   batch_size=256, shuffle=False, num_workers=0)

optim = torch.optim.Adam(model.parameters(), lr=1e-3)
crit  = nn.CrossEntropyLoss()
EPOCHS = 10

for ep in range(1, EPOCHS + 1):
    model.train()
    tot_loss = tot_correct = tot = 0
    for X, y in train_loader:
        X, y = X.to(device), y.to(device)
        logits = model(X)
        loss = crit(logits, y)
        optim.zero_grad(); loss.backward(); optim.step()
        tot_loss    += loss.item() * y.size(0)
        tot_correct += (logits.argmax(1) == y).sum().item()
        tot         += y.size(0)
    tr_loss = tot_loss / tot; tr_acc = tot_correct / tot

    model.eval()
    vc = vt = 0
    with torch.no_grad():
        for X, y in val_loader:
            X, y = X.to(device), y.to(device)
            vc += (model(X).argmax(1) == y).sum().item(); vt += y.size(0)
    print(f'epoch {ep:2d}  train_loss={tr_loss:.4f}  train_acc={tr_acc:.3f}  val_acc={vc/vt:.3f}')

epoch  1  train_loss=0.4703  train_acc=0.843  val_acc=0.777
epoch  2  train_loss=0.4347  train_acc=0.855  val_acc=0.774
epoch  3  train_loss=0.4048  train_acc=0.866  val_acc=0.775
epoch  4  train_loss=0.3745  train_acc=0.877  val_acc=0.766
epoch  5  train_loss=0.3489  train_acc=0.887  val_acc=0.778
epoch  6  train_loss=0.3201  train_acc=0.897  val_acc=0.778
epoch  7  train_loss=0.2930  train_acc=0.905  val_acc=0.766
epoch  8  train_loss=0.2624  train_acc=0.917  val_acc=0.777
epoch  9  train_loss=0.2351  train_acc=0.927  val_acc=0.765
epoch 10  train_loss=0.2111  train_acc=0.936  val_acc=0.771


## Per-class validation accuracy

In [9]:
model.eval()
correct = torch.zeros(N_CLASSES)
total   = torch.zeros(N_CLASSES)
with torch.no_grad():
    for X, y in val_loader:
        X, y = X.to(device), y.to(device)
        pred = model(X).argmax(1)
        for c in range(N_CLASSES):
            mask = (y == c)
            total[c]   += mask.sum().item()
            correct[c] += (pred[mask] == c).sum().item()

for c in range(N_CLASSES):
    if total[c] > 0:
        print(f'{IDX2LABEL[c]:>9}: {correct[c].item():>6.0f}/{total[c].item():<6.0f}  acc={correct[c].item()/total[c].item():.3f}')
    else:
        print(f'{IDX2LABEL[c]:>9}: no samples')

        a:   1027/1472    acc=0.698
        e:    695/1248    acc=0.557
        i:    666/978     acc=0.681
       i2:    324/595     acc=0.545
        o:    897/1249    acc=0.718
        u:    313/508     acc=0.616
        y: no samples
      eo5:     62/144     acc=0.431
      oc5:    129/265     acc=0.487
non_vowel:   6320/7069    acc=0.894
